In [ ]:
!pwd

In [ ]:
%cd ../..

In [ ]:
!pwd

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only
from trl import SFTConfig, SFTTrainer
import json
from transformers import EarlyStoppingCallback
import random

import os
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"

random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
HYPERPARAMS = {
    "MODEL_NAME": "unsloth/Qwen3.8-27B",
    "MAX_LEN": 918, #based on token length analysis
    "LOAD_IN_4BIT": True,
    "BATCH_SIZE":16,
    "GRAD_ACC": 2,
    "EPOCHS": 5,
    "LR": 2.4e-4,
    "LOG_STEPS": 10,
    "SAVE_STEPS": 100,
    "SAVE_LIMIT": 3,
    "EVAL_STEPS": 100,
    "WEIGHT_DECAY": 0.00465,
    "WARMUP_RATIO": 0.05,
    "MAX_GRAD_NORM": 1.0,

    "LORA_R": 32,
    "LORA_ALPHA": 64,
    "LORA_DROPOUT": 0.1,
    
    "ES_THRESHOLD": 0.001,
    "ES_PATIENCE": 5,
    
    "DATA_FILE_PATH": "Files/v3_gitapress_final.csv",
    "OUTPUT_DIR": "Trained_Models/Qwen/Qwen38-27B-0shotTrained",

}

In [ ]:
os.makedirs(HYPERPARAMS["OUTPUT_DIR"], exist_ok=True)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HYPERPARAMS["MODEL_NAME"],
    max_seq_length=HYPERPARAMS["MAX_LEN"],
    load_in_4bit=HYPERPARAMS["LOAD_IN_4BIT"],
    full_finetuning = False
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Reaches the linear_attn (gated delta) modules too
    finetune_mlp_modules       = True,  # Should leave on always!
    
    r=HYPERPARAMS["LORA_R"],
    lora_alpha=HYPERPARAMS["LORA_ALPHA"],
    lora_dropout=HYPERPARAMS["LORA_DROPOUT"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
ds = load_dataset('csv', data_files=HYPERPARAMS["DATA_FILE_PATH"])["train"]

train_ds = ds.filter(lambda x: x["split"] == "train")
val_ds = ds.filter(lambda x: x["split"] == "val")
print(f"Train: {len(train_ds)}")
print(f"Val: {len(val_ds)}")

In [ ]:
from collections import Counter
train_dist = Counter(train_ds["meter_cd"])
val_dist = Counter(val_ds["meter_cd"])
print("Train distribution:")
for meter, count in sorted(train_dist.items()):
    print(f"{meter}: {count}")
print("\nValidation distribution:")
for meter, count in sorted(val_dist.items()):
    print(f"{meter}: {count}")

In [ ]:
def format_and_tokenize(batch):
    texts = []

    for prompt, hi, sa in zip(
        batch["prompt"], batch["hi"], batch["sa"]
    ):
        messages = [
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": f"Meaning:\n{hi}\n\nGenerate the Sanskrit verse.",
            },
            {"role": "assistant", "content": sa},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
            preserve_thinking=False,
        )

        text = text.replace("<think>\n\n</think>\n\n", "")

        texts.append(text)

    return {"text": texts}

In [ ]:
train_ds = train_ds.map(format_and_tokenize, batched=True)
val_ds = val_ds.map(format_and_tokenize, batched=True)

In [ ]:
print(train_ds[0]['text'])

In [ ]:
messages = [
    {"role": "system", "content": "Generate a Sanskrit verse."},
    {"role": "user", "content": "Meaning: test"},
    {"role": "assistant", "content": "ब्रह्मादयः सुरगणाः।"},
]

In [ ]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False,
    preserve_thinking=False,
)

In [ ]:
print(repr(text))
print("Has think block:", "<think>" in text)